# Knowledge Distillation Basics

## What is Knowledge Distillation?

Knowledge distillation is a technique where a large, complex model (the "teacher") trains a smaller, simpler model (the "student") to perform the same task. The student learns not just from the original training data, but also from how the teacher model makes predictions.

Think of it like this: instead of learning from scratch, the student gets to see both the correct answers AND how an expert thinks about the problem.

## Why Use Knowledge Distillation?

**The Problem**: Large models are accurate but slow and expensive to run. Small models are fast but less accurate.

**The Solution**: Train a small model to mimic a large model's behavior. The result is a model that's almost as accurate as the large one, but much faster and cheaper to use.

**Real Benefits**:
- Smaller models that fit on phones or edge devices
- Faster inference times (milliseconds instead of seconds)
- Lower computational costs
- Easier deployment in production

## Setup: Import Required Libraries

Let's start by importing the libraries we'll need for our examples.

In [ ]:
import os

import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
from torchvision import datasets
import numpy as np
import matplotlib.pyplot as plt

print("PyTorch version:", torch.__version__)
print("Setup complete!")

### Reproducibility

`SEED` fixes the teacher's initial weights and the order of its training batches.

In [ ]:
SEED = 0
torch.manual_seed(SEED)
# downloads land outside the repository, set KD_DATA_DIR to put them somewhere else
DATA_DIR = os.environ.get('KD_DATA_DIR', os.path.join(os.path.expanduser('~'), '.cache', 'kd-data'))

## Core Concept 1: Hard vs Soft Targets

The key innovation in knowledge distillation is using soft targets instead of just hard labels.

- **Hard label**: A single answer (e.g., "cat")
- **Soft label**: Probabilities for all answers (e.g., 90% cat, 5% dog, 3% bird, 2% fish)

The soft label tells us the teacher is very confident it's a cat, but there's a tiny chance it could be a dog. This uncertainty is valuable information!

In [ ]:
# a one hot label only names the right class, a teacher's distribution also says how wrong the others are
classes = ['cat', 'dog', 'bird', 'fish']

# hard target: all probability mass on the true class
hard_target = np.array([1, 0, 0, 0])
print("Hard Target (one-hot):")
for i, prob in enumerate(hard_target):
    print(f"  {classes[i]}: {prob}")

print("\n" + "="*40 + "\n")

In [ ]:
# soft target: an invented teacher output, the 0.05 on dog says this cat looks a little like a dog
soft_target = np.array([0.90, 0.05, 0.03, 0.02])
print("Soft Target (from teacher):")
for i, prob in enumerate(soft_target):
    print(f" {classes[i]}: {prob}")

print("\nThe soft target reveals that the image has some dog-like features,")
print("even though the teacher is confident it's a cat!")

## Core Concept 2: Temperature Scaling

Temperature is a parameter that controls how "soft" the probability distribution becomes. Higher temperature makes the probabilities more uniform, revealing more information about what the teacher learned.

The formula is: `softmax(logits / temperature)`

- **Temperature = 1**: Normal probabilities (sharp)
- **Temperature > 1**: Softer probabilities (more uniform)
- **Temperature < 1**: Sharper probabilities (more peaked)

In [ ]:
# softmax(z / t): a larger t flattens the distribution but never changes the ranking
logits = torch.tensor([10.0, 2.0, 1.0, 0.5])

temperatures = [1, 3, 5, 10]

print("Effect of Temperature on Probability Distribution:")
print("="*60)

for T in temperatures:
    probs = F.softmax(logits / T, dim=0)
    print(f"\nTemperature = {T}:")
    for i, prob in enumerate(probs):
        print(f"  Class {i}: {prob:.4f}")
    
print("\nNotice: At T=1, class 0 takes almost all the mass. At T=10 the")
print("ranking of classes 1, 2 and 3 behind it becomes visible.")

### Visualizing Temperature Effects

Let's visualize how temperature affects the probability distribution:

In [ ]:
logits = torch.tensor([10.0, 2.0, 1.0, 0.5])
temperatures = [1, 2, 5, 10]
classes = ['Class 0', 'Class 1', 'Class 2', 'Class 3']

fig, axes = plt.subplots(2, 2, figsize=(12, 8))
axes = axes.flatten()

for idx, T in enumerate(temperatures):
    probs = F.softmax(logits / T, dim=0).numpy()
    axes[idx].bar(classes, probs, color='skyblue', edgecolor='black')
    axes[idx].set_title(f'Temperature = {T}', fontsize=12, fontweight='bold')
    axes[idx].set_ylabel('Probability')
    axes[idx].set_ylim([0, 1])
    axes[idx].grid(axis='y', alpha=0.3)
    
    for i, v in enumerate(probs):
        axes[idx].text(i, v + 0.02, f'{v:.3f}', ha='center', fontsize=9)

plt.tight_layout()
plt.savefig('temperature_effect.png', dpi=100, bbox_inches='tight')
plt.show()

In [ ]:
print("Higher temperature spreads probability mass to the other classes,")
print("the part of the distribution Hinton et al. (2015) call 'dark knowledge'.")

## Example: Simple Teacher and Student Models

Let's create simple teacher and student models to see the size difference:

In [ ]:
# teacher: wide mlp, about 13x the student's parameters
class TeacherModel(nn.Module):
    def __init__(self, input_size=784, num_classes=10):
        super(TeacherModel, self).__init__()
        self.network = nn.Sequential(
            nn.Linear(input_size, 1024),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(1024, 512),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(512, 256),
            nn.ReLU(),
            nn.Linear(256, num_classes)
        )
    
    def forward(self, x):
        return self.network(x)

In [ ]:
# student: small mlp, the model you would deploy
class StudentModel(nn.Module):
    def __init__(self, input_size=784, num_classes=10):
        super(StudentModel, self).__init__()
        self.network = nn.Sequential(
            nn.Linear(input_size, 128),
            nn.ReLU(),
            nn.Linear(128, 64),
            nn.ReLU(),
            nn.Linear(64, num_classes)
        )
    
    def forward(self, x):
        return self.network(x)

In [ ]:
teacher = TeacherModel()
student = StudentModel()

teacher_params = sum(p.numel() for p in teacher.parameters())
student_params = sum(p.numel() for p in student.parameters())

In [ ]:
print("Model Comparison:")
print("="*50)
print(f"Teacher Parameters: {teacher_params:,}")
print(f"Student Parameters: {student_params:,}")
print(f"Compression Ratio: {teacher_params/student_params:.1f}x smaller")
print(f"Parameter Reduction: {(1 - student_params/teacher_params)*100:.1f}%")

## Example: Teacher's Predictions vs Hard Labels

Let's see how a teacher model's soft predictions differ from hard labels on a sample.
The teacher above is trained for two epochs on MNIST, then shown the test digit it is least sure about among the first 1,000.

In [ ]:
def load_mnist(train):
    # flattened 28x28 digits, standardised with the usual mnist mean and standard deviation
    raw = datasets.MNIST(DATA_DIR, train=train, download=True)
    x = (raw.data.float() / 255.0 - 0.1307) / 0.3081
    return x.view(len(x), -1), raw.targets.clone()

train_X, train_y = load_mnist(train=True)
test_X, test_y = load_mnist(train=False)

In [ ]:
train_loader = DataLoader(TensorDataset(train_X, train_y), batch_size=128, shuffle=True,
                          generator=torch.Generator().manual_seed(SEED))
optimizer = optim.Adam(teacher.parameters(), lr=0.001)
teacher.train()
for epoch in range(2):
    for batch_X, batch_y in train_loader:
        optimizer.zero_grad()
        F.cross_entropy(teacher(batch_X), batch_y).backward()
        optimizer.step()

In [ ]:
teacher.eval()
with torch.no_grad():
    test_accuracy = (teacher(test_X).argmax(dim=1) == test_y).float().mean().item()
print(f"teacher test accuracy after 2 epochs: {test_accuracy * 100:.2f}%")

In [ ]:
# the least confident of the first 1,000 test digits, where the soft targets have the most to say
with torch.no_grad():
    candidate_logits = teacher(test_X[:1000])
sample = int(F.softmax(candidate_logits, dim=1).max(dim=1).values.argmin())
true_label = int(test_y[sample])

teacher_logits = candidate_logits[sample:sample + 1]
teacher_probs_T1 = F.softmax(teacher_logits, dim=1)
teacher_probs_T5 = F.softmax(teacher_logits / 5.0, dim=1)

hard_label = torch.zeros(10)
hard_label[true_label] = 1.0

In [ ]:
fig, axes = plt.subplots(1, 4, figsize=(18, 4))

axes[0].imshow(test_X[sample].view(28, 28).numpy(), cmap='gray')
axes[0].set_title(f'Test digit {sample}', fontweight='bold')
axes[0].axis('off')

panels = [('Hard Label (Ground Truth)', hard_label, 'green'),
          ('Teacher Predictions (T=1)', teacher_probs_T1[0], 'blue'),
          ('Teacher Predictions (T=5)', teacher_probs_T5[0], 'orange')]
for ax, (title, probs, color) in zip(axes[1:], panels):
    ax.bar(range(10), probs.numpy(), color=color, alpha=0.7)
    ax.set_title(title, fontweight='bold')
    ax.set_xlabel('Digit')
    ax.set_xticks(range(10))
    ax.set_ylabel('Probability')
    ax.set_ylim([0, 1])
    ax.grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.savefig('hard_vs_soft_targets.png', dpi=100, bbox_inches='tight')
plt.show()

In [ ]:
ranked = teacher_probs_T1[0].argsort(descending=True)
print(f"true digit: {true_label}, teacher's top two at T=1: {int(ranked[0])} ({teacher_probs_T1[0, ranked[0]]:.3f}), "
      f"{int(ranked[1])} ({teacher_probs_T1[0, ranked[1]]:.3f}), "
      f"so the teacher is {'right' if int(ranked[0]) == true_label else 'wrong'} on this digit")
print("The hard label only names the true digit, the teacher's distribution also says")
print("which other digit this image resembles, and T=5 makes that visible across all ten classes.")

## The Basic Distillation Process

Here's the basic workflow for knowledge distillation:

1. **Train or load a teacher model** - Get a large, accurate model
2. **Design a student model** - Create a smaller architecture
3. **Generate soft targets** - Run teacher on data with high temperature
4. **Train the student** - Use both soft targets and hard labels
5. **Evaluate** - Compare student to teacher performance

Let's see a simple example of the training loop structure:

In [ ]:
# pseudocode for one distillation run, printed rather than executed
print("Basic Knowledge Distillation Training Loop:")
print("""
for epoch in range(num_epochs):
    for batch_data, batch_labels in dataloader:
        
        # Step 1: Get teacher's soft predictions
        with torch.no_grad():
            teacher_logits = teacher_model(batch_data)
            teacher_probs = softmax(teacher_logits / temperature)
        
        # Step 2: Get student's predictions
        student_logits = student_model(batch_data)
        student_probs = softmax(student_logits / temperature)
        
        # Step 3: Calculate loss
        # - Distillation loss: KL(teacher || student) on the softened outputs,
        #   times T^2 so its gradients keep their scale as T changes
        # - Student loss: Match true labels at T=1
        distill_loss = kl_divergence(teacher_probs, student_probs) * temperature**2
        student_loss = cross_entropy(student_logits, batch_labels)
        
        total_loss = alpha * distill_loss + (1-alpha) * student_loss
        
        # Step 4: Update student only (teacher stays frozen)
        optimizer.zero_grad()
        total_loss.backward()
        optimizer.step()
""")